### Calculation of the ratio S/N

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
#In[2]:
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprosess

In [ ]:
# import src.slurm_cluster as scluster
# client, scluster = scluster.init_dask_slurm_cluster()

### Input both the forced and ICV_std trend data 

In [ ]:
dir_in= '/work/mh0033/m301036/OBS_LPS_revision/docs/data/Revision_check/pseudo_obs_check/trend/'

MPI_ESM_ICV_ds = xr.open_mfdataset(dir_in + 'MPI_ESM_tas_ICV_trend_wrt_MMEM.nc')
print(MPI_ESM_ICV_ds)

In [ ]:
# dir_ICV = '/work/mh0033/m301036/Land_surf_temp/Disentangling_OBS_SAT_trend/Revision_check/pseudo_obs_check/data/trend/'
# # Input multiple runs ICV into one dataset with new variable dimension 'run'
# variable_indices = np.arange(0, 50, 1).astype(str)
# segment_lengths = 30
# # 
# # Initialize an empty list to collect datasets
# datasets = []

# for var in variable_indices:
#     print(f"Processing run {var}")
#     file = dir_ICV + f'ICV_trend_run{var}_wrt_IPSL_CM6A.nc'
    
#     # Open the dataset for the current run
#     try:
#         ds = xr.open_mfdataset(file, engine="netcdf4",combine='by_coords')
#     except FileNotFoundError:
#         print(f"File not found: {file}")
#         continue
    
#     datasets.append(ds)

# # Combine all datasets along the `run` dimension
# if datasets:
#     MPI_ESM_ICV_ds = xr.concat(datasets, dim="run")
#     # change the run dimension range from 0-49 to 1-50
#     MPI_ESM_ICV_ds['run'] = np.arange(1, 51, 1)
#     print("Datasets successfully combined!")
# else:
#     print("No datasets were combined.")

In [ ]:
MPI_ESM_ICV_ds

In [ ]:
# calculate the standard deviation of the ICV
# define function to calculate the standard deviation of the trend pattern of each interval of segments
def std_trend_pattern(data):
    """
    data: 3D array with dimensions [segment, lat, lon]
    segment_number: number of segments
    """
    
    # Calculate the standard deviation of the trend pattern of each interval of segments
    std_trend_pattern = np.nanstd(data, axis=0)
    
    return std_trend_pattern

In [ ]:
# === Compute standard deviation across segments for trend_30 ===
std_pattern = MPI_ESM_ICV_ds['trend_30'].std(dim="segment")

In [ ]:
std_pattern

In [ ]:
MPI_ESM_ICV_noise_ds = xr.Dataset({
    "std_trend_30": std_pattern
})

In [ ]:
# save the standard deviation of the trend pattern of each interval of segments to a NetCDF file
output_file = dir_in + 'MPI_ESM_ICV_noise_std_trend_pattern_MMEM.nc'
MPI_ESM_ICV_noise_ds.to_netcdf(output_file)
print(f"Standard deviation of the trend pattern of each interval of segments saved to: {output_file}")

In [ ]:
MPI_ESM_ICV_noise_ds

In [ ]:
plt.rcParams['figure.figsize'] = (8, 10)
plt.rcParams['font.size'] = 16
# plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.labelsize'] = 16
plt.rcParams['ytick.direction'] = 'out'
plt.rcParams['ytick.minor.visible'] = True
plt.rcParams['ytick.major.right'] = True
plt.rcParams['ytick.right'] = True
plt.rcParams['xtick.bottom'] = True
plt.rcParams['savefig.transparent'] = True

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm, ListedColormap

In [ ]:
def plot_trend(trend_data, lats, lons, levels=None, extend=None, cmap=None, 
                                 title="", ax=None, show_xticks=False, show_yticks=False):
    """
    Plot the trend spatial pattern using Robinson projection with significance overlaid.

    Parameters:
    - trend_data: 2D numpy array with the trend values.
    - lats, lons: 1D arrays of latitudes and longitudes.
    - p_values: 2D array with p-values for each grid point.
    - GMST_p_values: 2D array with GMST p-values for each grid point.
    - title: Title for the plot.
    - ax: Existing axis to plot on. If None, a new axis will be created.
    - show_xticks, show_yticks: Boolean flags to show x and y axis ticks.
    
    Returns:
    - contour_obj: The contour object from the plot.
    """
# Create a new figure/axis if none is provided
    if ax is None:
        fig, ax = plt.subplots(figsize=(20, 15), subplot_kw={'projection': ccrs.Robinson()})
        ax.set_global()
        
    contour_obj = ax.contourf(lons, lats, trend_data, levels=levels, extend=extend, cmap=cmap, transform=ccrs.PlateCarree(central_longitude=0))
    # Plot significance masks with different hatches
    # ax.contourf(lons, lats, significance_mask, levels=[0.05, 1.0],hatches=['///'], colors='none', transform=ccrs.PlateCarree())

    ax.coastlines(resolution='110m')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, linewidth=1, color='gray', alpha=0.35)

    # Disable labels on the top and right of the plot
    gl.top_labels = False
    gl.right_labels = False

    # Enable labels on the bottom and left of the plot
    gl.bottom_labels = show_xticks
    gl.left_labels = show_yticks
    gl.xformatter = cticker.LongitudeFormatter()
    gl.yformatter = cticker.LatitudeFormatter()
    gl.xlabel_style = {'size': 16}
    gl.ylabel_style = {'size': 16}
    
    if show_xticks:
        gl.bottom_labels = True
    if show_yticks:
        gl.left_labels = True
    
    ax.set_title(title, loc='center', fontsize=24, pad=5.0)

    return contour_obj

In [ ]:
# define an asymmetric colormap
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.colors import BoundaryNorm

intervals = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.4, 1.6, 1.8, 2.0]

# Normalizing the intervals to [0, 1]
min_interval = min(intervals)
max_interval = max(intervals)
normalized_intervals = [(val - min_interval) / (max_interval - min_interval) for val in intervals]

In [ ]:
import seaborn as sns
import palettable
import matplotlib.colors as mcolors

cmap = mcolors.ListedColormap(palettable.cmocean.sequential.Amp_20.mpl_colors)

In [ ]:
MPI_ESM_ICV_noise_ds['std_trend_30']

In [ ]:
r1_std_trend_10 = MPI_ESM_ICV_noise_ds['std_trend_30'].sel(run=1)

In [ ]:
r1_std_trend_10

In [ ]:
lat = MPI_ESM_ICV_noise_ds['lat'].values
lon = MPI_ESM_ICV_noise_ds['lon'].values

titles = ["30yr"]

import cartopy.util as cutil
# levels = np.arange(-0.2, 0.25, 0.025)
# Define the GridSpec
fig,ax = plt.subplots(1, 1, figsize=(15, 12), subplot_kw={'projection': ccrs.Robinson(180)})

levels = np.arange(0.0, 1.1, 0.1)

trend_data_10yr = r1_std_trend_10.values
trend_with_cyclic_10yr, lon_with_cyclic = cutil.add_cyclic_point(trend_data_10yr, coord=lon)
contour_obj = plot_trend(trend_with_cyclic_10yr, lat, lon_with_cyclic,levels=levels,extend='max',
                    cmap=cmap,title=titles[0], ax=ax, show_xticks = True, show_yticks = True)

cbar_ax = fig.add_axes([0.3, 0.13, 0.5, 0.025])
cbar = plt.colorbar(contour_obj, cax=cbar_ax, orientation='horizontal', extend='max')
cbar.ax.tick_params(labelsize=18)
cbar.set_label('°C/decade', fontsize=22)

plt.tight_layout()
fig.savefig('MPI_ESM_run1_segmentes_ICV_trend_std_patterns_30yr_MMEM.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# client.close()
# scluster.close()

### 